# Feature engineering: train/test split first

Two features for the SaiPrice price model (CLAUDE.md §13):

| Feature | Level | Leakage risk |
|---|---|---|
| `listing_age` | row | none, computed per row from `posted_date` |
| `district_median_price` | group | **yes**, fitted on train only |

Order is deliberate: **split, then fit district statistics on train, then map those
onto test.** Computing a district median over the full frame before splitting puts
test-set prices into a train-set feature, and the resulting score is inflated by an
amount you cannot measure afterwards.

### A third feature was cut, and it is the more interesting story

`price_per_sqm_vs_district_avg` was built here and removed on 2026-07-28. It was
`(price / area_sqm) / district_avg`, which means `ratio × area_sqm × district_avg`
reconstructs `price` **exactly** — verified on 881 of 881 train rows. Since
`area_sqm` was a feature and `district_avg` is a function of `district`, also a
feature, the target was recoverable by multiplication. Test R² went 0.57 → 0.96 and
the fitted coefficient on `log(ratio)` came out at 0.98: the model was not
estimating price, it was solving an identity.

Splitting correctly does not protect you from this. Train/test leakage and target
leakage are different axes, and the split ordering below only addresses the first.
The feature is also uncomputable at scoring time, where `price` is the unknown.

## 1. Environment smoke test

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

numpy        2.5.1
pandas       3.0.5
scikit-learn 1.9.0


## 2. Load the cleaned training set

In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "saiprice.settings")
# ipykernel runs a live event loop, which trips Django's sync-DB-in-async guard.
# Safe here: build_training_rows is read-only (listings/ml/dataset.py docstring).
os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")

import django

django.setup()

from listings.ml.dataset import build_training_rows

rows, clean_stats = build_training_rows()
df = pd.DataFrame(rows)

for name, value in clean_stats.items():
    print(f"{name:32} {value}")
print()
print(df.dtypes.to_string())
df.head()

rows_before                      1175
dropped_not_sale                 1
dropped_missing_or_nonpositive   0
dropped_missing_posted_date      0
dropped_whole_building           40
dropped_duplicate                32
rows_after                       1102

id                 int64
district             str
property_type        str
area_sqm         float64
price            float64
posted_date       object


,id,district,property_type,area_sqm,price,posted_date
0,1,Quận 7,apartment,71.5,6.280000e+09,2026-07-04
1,3,Quận Tân Bình,house,39.0,7.980000e+09,2026-07-06
2,5,Huyện Nhà Bè,apartment,78.0,3.200000e+09,2026-07-05
3,7,Quận 7,apartment,85.0,3.590000e+09,2026-07-13
4,8,Huyện Bình Chánh,apartment,52.0,1.600000e+09,2026-07-10


## 3. Split — before any group-level statistic

Nothing above this cell has touched `district`. `train_test_split` runs on the raw
cleaned frame, so the two frames are independent before a single median is computed.

In [3]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

train, test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

assert not (set(train.index) & set(test.index)), "train/test overlap"
print(f"train {len(train)} rows")
print(f"test  {len(test)} rows")

train 881 rows
test  221 rows


## 4a. `listing_age` — row-level, run on each frame independently

No group statistic, so there is nothing to leak. `AS_OF` is pinned rather than
defaulted to today, otherwise the feature drifts by a day on every rerun and the
notebook stops being reproducible.

In [4]:
from listings.ml.features import (
    MIN_DISTRICT_ROWS,
    district_median_price,
    fit_district_stats,
    listing_age,
)

AS_OF = pd.Timestamp("2026-07-28")

train = train.assign(listing_age=listing_age(train, AS_OF))
test = test.assign(listing_age=listing_age(test, AS_OF))

print(train["listing_age"].describe().to_string())

count    881.000000
mean       2.913734
std        3.408453
min        0.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       45.000000


## 4b. Fit district statistics — **train only**

This is the one cell that reads a group statistic out of the data. It sees `train`
and never `test` or `df`.

Districts with fewer than `MIN_DISTRICT_ROWS` train rows have their median replaced
by the train-wide median. With one or three rows, a district's "median" is largely
the listing being scored reading back its own price — in-fold target encoding, a
milder version of the mistake that killed the third feature. They keep their place
in the index; only the value changes.

In [5]:
district_stats = fit_district_stats(train)
counts = train["district"].value_counts()
thin = sorted(counts[counts < MIN_DISTRICT_ROWS].index)

print(f"districts seen in train: {len(district_stats)}")
print(f"collapsed as thin (< {MIN_DISTRICT_ROWS} train rows): {thin or 'none'}")
for d in thin:
    own = train.loc[train["district"] == d, "price"].median()
    print(f"  {d:20} {counts[d]:>3} rows   own median {own:>15,.0f}"
          f" -> {district_stats.loc[d, 'median_price']:>15,.0f}")
district_stats.sort_values("median_price", ascending=False).head(10)

districts seen in train: 21
collapsed as thin (< 10 train rows): ['Huyện Cần Giờ', 'Huyện Hóc Môn']
  Huyện Cần Giờ          1 rows   own median   2,000,000,000 ->   4,300,000,000
  Huyện Hóc Môn          3 rows   own median   1,900,000,000 ->   4,300,000,000


,median_price,train_median_price
district,,
Quận 1,9.200000e+09,4.300000e+09
Quận Phú Nhuận,6.200000e+09,4.300000e+09
Quận Bình Thạnh,5.900000e+09,4.300000e+09
Quận 4,5.500000e+09,4.300000e+09
Quận 7,5.150000e+09,4.300000e+09
Thành phố Thủ Đức,4.675000e+09,4.300000e+09
Quận Gò Vấp,4.600000e+09,4.300000e+09
Quận 3,4.600000e+09,4.300000e+09
Huyện Nhà Bè,4.400000e+09,4.300000e+09


## 4c. Map the fitted statistics onto both frames

Same `district_stats` object passed to both calls. Test rows are looked up by
district key, never recomputed.

In [6]:
train = train.assign(
    district_median_price=district_median_price(train, district_stats),
)
test = test.assign(
    district_median_price=district_median_price(test, district_stats),
)

FEATURES_NEW = ["listing_age", "district_median_price"]
test[["district", "price", "area_sqm", *FEATURES_NEW]].head()

,district,price,area_sqm,listing_age,district_median_price
309,Quận 4,5.500000e+09,92.0,0,5.500000e+09
1040,Thành phố Thủ Đức,5.500000e+09,82.0,1,4.675000e+09
381,Quận 7,8.700000e+09,129.0,6,5.150000e+09
497,Quận 6,2.400000e+09,50.0,1,3.225000e+09
914,Quận 6,3.650000e+09,94.0,3,3.225000e+09


## 5. Verify

### 5a. The split is actually doing something

If fitting on the full frame gave the same numbers, the split would be decorative.
It does not, which is the leak that was avoided.

In [7]:
full_stats = fit_district_stats(pd.concat([train, test]))
shift = (
    (district_stats["median_price"] - full_stats["median_price"]).abs()
    / full_stats["median_price"]
)
print(f"districts whose median price would move if fitted on train+test: "
      f"{(shift > 0.001).sum()} / {len(shift)}")
print(f"largest shift: {shift.max():.1%}")

districts whose median price would move if fitted on train+test: 15 / 21
largest shift: 31.1%


### 5b. Nulls and infs introduced by the new features

In [8]:
for name, frame in (("train", train), ("test", test)):
    nulls = frame[FEATURES_NEW].isna().sum()
    infs = int(np.isinf(frame[FEATURES_NEW].to_numpy(dtype="float64")).sum())
    print(f"--- {name} ({len(frame)} rows) ---")
    print(nulls.to_string())
    print(f"{'infs':32} {infs}")
    print()

--- train (881 rows) ---
listing_age              0
district_median_price    0
infs                             0

--- test (221 rows) ---
listing_age              0
district_median_price    0
infs                             0



### 5c. Unseen districts

A district in test but absent from train has no fitted median, so those rows take
`train_median_price` — the train-wide median computed inside `fit_district_stats`
from the train frame alone, which is what keeps the fallback leak-free. Thin
districts (§4b) resolve to that same value.

The count is still reported rather than silently absorbed: a rising number here
means the split is starving districts that were already thin, which is a data
problem the imputation would otherwise hide.

In [9]:
unseen = sorted(set(test["district"]) - set(district_stats.index))
affected = int(test["district"].isin(unseen).sum())

print(f"districts in test but not train: {unseen or 'none'}")
print(f"test rows affected: {affected} / {len(test)}")
if unseen:
    print()
    print("train-set size of each unseen district (0 by definition):")
    for d in unseen:
        print(f"  {d:24} full-frame rows: {int((df['district'] == d).sum())}")

districts in test but not train: none
test rows affected: 0 / 221


### 5d. Feature tests

In [10]:
import subprocess

result = subprocess.run(
    [sys.executable, "manage.py", "test",
     "listings.tests.test_features", "listings.tests.test_dataset", "-v", "2"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stderr or result.stdout)
assert result.returncode == 0, "feature tests failed"

Creating test database for alias 'default' ('test_saiprice_db')...

test_carries_posted_date_for_the_listing_age_feature (listings.tests.test_dataset.BuildTrainingRowsTests.test_carries_posted_date_for_the_listing_age_feature) ... ok
test_drops_each_dirty_row_and_counts_it (listings.tests.test_dataset.BuildTrainingRowsTests.test_drops_each_dirty_row_and_counts_it) ... ok
test_drops_listing_with_null_posted_date (listings.tests.test_dataset.BuildTrainingRowsTests.test_drops_listing_with_null_posted_date) ... ok
test_drops_repost_duplicates_keeping_first_seen (listings.tests.test_dataset.BuildTrainingRowsTests.test_drops_repost_duplicates_keeping_first_seen) ... ok
test_keeps_clean_sale_listing (listings.tests.test_dataset.BuildTrainingRowsTests.test_keeps_clean_sale_listing) ... ok
test_no_nulls_in_any_model_column (listings.tests.test_dataset.BuildTrainingRowsTests.test_no_nulls_in_any_model_column) ... ok
test_maps_train_median_and_falls_back_for_unseen_district (listings.tests.test_f